In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

In [0]:
df = spark.table("workspace.bronze.geolocation")


In [0]:
string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]

df = df.select(
    *[F.trim(F.col(c)).alias(c) if c in string_cols else F.col(c) for c in df.columns]
)

In [0]:
df = (
    df
    .filter(F.col("geolocation_zip_code_prefix").isNotNull())
    # Deduplicate by zip_prefix to prevent row explosion during joins
    .dropDuplicates(["geolocation_zip_code_prefix"])
    
    # City: Match Customer logic (Lowercase + Remove Accents)
    .withColumn("geolocation_city", F.lower(F.col("geolocation_city")))
    .withColumn("geolocation_city", 
                F.translate(F.col("geolocation_city"), 
                            "áàâãäéèêëíìîïóòôõöúùûüç", 
                            "aaaaaeeeeiiiiooooouuuuc"))
    
    # State: Standardize to Uppercase (2 chars)
    .withColumn("geolocation_state", F.upper(F.substring(F.col("geolocation_state"), 1, 2)))
    
    # Rename Columns
    .withColumnRenamed("geolocation_zip_code_prefix", "zip_prefix")
    .withColumnRenamed("geolocation_lat", "latitude")
    .withColumnRenamed("geolocation_lng", "longitude")
    
    # Zip Code: Match Customer logic (Padded String "00000")
    .withColumn("zip_prefix", F.lpad(F.col("zip_prefix").cast("string"), 5, "0"))
    
    # Data type safety
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("longitude", F.col("longitude").cast("double"))
    
    .withColumn("silver_processed_time", F.current_timestamp())
)

In [0]:
df.select("zip_prefix", "latitude", "longitude", "geolocation_city").show(5)

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.geolocation")
)